# 01 - Exploration : régularité mensuelle TGV par liaison

Objectif : connaître la source avant d'écrire le pipeline. Chaque anomalie relevée ici deviendra un test dbt (étape 6).
Les conclusions sont reportées dans `docs/exploration.md`.

Source : [data.gouv.fr - Régularité mensuelle TGV par liaisons](https://www.data.gouv.fr/datasets/regularite-mensuelle-tgv-par-liaisons) (AQST / SNCF, licence ODbL).

In [1]:
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

URL = "https://www.data.gouv.fr/api/1/datasets/r/91fe399d-cafa-4e72-8ba3-56d8717fdad4"

## Chargement

On applique la même normalisation `snake_case` que l'ingestion (`include/ingestion/sncf.py`) :
les noms de colonnes ci‑dessous sont donc ceux qu'on retrouvera dans la table bronze.

In [2]:
def to_snake(name: str) -> str:
    """'Gare de départ' -> 'gare_de_depart'."""
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()


raw = pd.read_csv(URL, sep=";")
df = raw.rename(columns=to_snake)
pd.DataFrame({"colonne_source": raw.columns, "colonne_bronze": df.columns})

,colonne_source,colonne_bronze
0,Date,date
1,Service,service
2,Gare de départ,gare_de_depart
3,Gare d'arrivée,gare_d_arrivee
4,Durée moyenne du trajet,duree_moyenne_du_trajet
5,Nombre de circulations prévues,nombre_de_circulations_prevues
6,Nombre de trains annulés,nombre_de_trains_annules
7,Commentaire annulations,commentaire_annulations
8,Nombre de trains en retard au départ,nombre_de_trains_en_retard_au_depart
9,Retard moyen des trains en retard au départ,retard_moyen_des_trains_en_retard_au_depart


Pour la suite, des alias courts sur les colonnes les plus utilisées.

In [3]:
KEY = ["date", "service", "gare_de_depart", "gare_d_arrivee"]
PREV = "nombre_de_circulations_prevues"
ANN = "nombre_de_trains_annules"
RET_ARR = "nombre_de_trains_en_retard_a_l_arrivee"
RET_DEP = "nombre_de_trains_en_retard_au_depart"
R15, R30, R60 = (f"nombre_trains_en_retard_{n}min" for n in (15, 30, 60))
CAUSES = [c for c in df.columns if c.startswith("prct_")]
COMMENTAIRES = [c for c in df.columns if c.startswith("commentaire")]

## Vue d'ensemble

In [4]:
print("Volume :", df.shape)
print("Période :", df["date"].min(), "->", df["date"].max(), f"({df['date'].nunique()} mois)")
df.dtypes

Volume : (12907, 26)
Période : 2018-01 -> 2026-09 (105 mois)


date                                                                                           str
service                                                                                        str
gare_de_depart                                                                                 str
gare_d_arrivee                                                                                 str
duree_moyenne_du_trajet                                                                      int64
nombre_de_circulations_prevues                                                               int64
nombre_de_trains_annules                                                                     int64
commentaire_annulations                                                                    float64
nombre_de_trains_en_retard_au_depart                                                         int64
retard_moyen_des_trains_en_retard_au_depart                                                float64
retard_moy

In [5]:
df.isna().mean().sort_values(ascending=False).head(8)

commentaire_annulations            1.000000
commentaire_retards_au_depart      1.000000
commentaire_retards_a_l_arrivee    0.945921
gare_de_depart                     0.000000
service                            0.000000
gare_d_arrivee                     0.000000
nombre_de_circulations_prevues     0.000000
nombre_de_trains_annules           0.000000
dtype: float64

In [6]:
df["service"].value_counts()

service
National         11339
International     1568
Name: count, dtype: int64

In [7]:
df.select_dtypes("number").describe().T

,count,mean,std,min,25%,50%,75%,max
duree_moyenne_du_trajet,12907.0,170.693112,87.726724,0.000000,99.000000,163.000000,223.000000,786.000000
nombre_de_circulations_prevues,12907.0,273.443635,184.117868,0.000000,151.000000,231.000000,365.000000,1104.000000
nombre_de_trains_annules,12907.0,8.273650,21.805356,0.000000,0.000000,2.000000,7.000000,297.000000
commentaire_annulations,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_de_trains_en_retard_au_depart,12907.0,86.703882,88.749586,0.000000,22.000000,52.000000,127.000000,598.000000
retard_moyen_des_trains_en_retard_au_depart,12907.0,12.598479,11.584859,0.000000,6.432137,10.696078,16.231349,316.188095
retard_moyen_de_tous_les_trains_au_depart,12907.0,3.218180,5.016502,-229.269444,1.250273,2.400417,4.094720,115.047390
commentaire_retards_au_depart,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_de_trains_en_retard_a_l_arrivee,12907.0,38.874797,32.454431,0.000000,16.000000,30.000000,53.000000,376.000000
retard_moyen_des_trains_en_retard_a_l_arrivee,12907.0,35.618496,15.611291,-40.109259,26.091113,33.895833,42.912828,299.600000


## Q1. Quelle est la granularité d'une ligne ? Y a‑t‑il des doublons ?

In [8]:
print("Doublons sur (date, service, gare_de_depart, gare_d_arrivee) :", df.duplicated(KEY).sum())
print("Doublons sur (date, gare_de_depart, gare_d_arrivee), sans le service :",
      df.duplicated(["date", "gare_de_depart", "gare_d_arrivee"]).sum())

Doublons sur (date, service, gare_de_depart, gare_d_arrivee) : 0
Doublons sur (date, gare_de_depart, gare_d_arrivee), sans le service : 42


Le service fait partie de la clé : certaines liaisons existent en `National` **et** en `International` le même mois.

In [9]:
od = ["date", "gare_de_depart", "gare_d_arrivee"]
double_service = df[df.duplicated(od, keep=False)]
double_service["date"].value_counts().sort_index()

date
2025-07    28
2025-08    28
2025-09    28
Name: count, dtype: int64

In [10]:
double_service.sort_values(od)[KEY + [PREV]].head(6)

,date,service,gare_de_depart,gare_d_arrivee,nombre_de_circulations_prevues
11055,2025-07,National,BELLEGARDE (AIN),PARIS LYON,22
11130,2025-07,International,BELLEGARDE (AIN),PARIS LYON,231
11078,2025-07,International,CHAMBERY CHALLES LES EAUX,PARIS LYON,96
11137,2025-07,National,CHAMBERY CHALLES LES EAUX,PARIS LYON,148
11102,2025-07,International,DIJON VILLE,PARIS LYON,186
11138,2025-07,National,DIJON VILLE,PARIS LYON,220


In [11]:
liaisons = df.groupby(["service", "gare_de_depart", "gare_d_arrivee"])["date"].agg(["count", "min", "max"])
print("Nombre de liaisons (service x départ x arrivée) :", len(liaisons))
print("Lignes par mois : min", df.groupby("date").size().min(), "/ max", df.groupby("date").size().max())
liaisons[liaisons["count"] < df["date"].nunique()].sort_values("count")

Nombre de liaisons (service x départ x arrivée) : 146
Lignes par mois : min 118 / max 135


count      min      max
service       gare_de_depart            gare_d_arrivee                                    
International BELLEGARDE (AIN)          PARIS LYON                     3  2025-07  2025-09
              CHAMBERY CHALLES LES EAUX PARIS LYON                     3  2025-07  2025-09
              DIJON VILLE               PARIS LYON                     3  2025-07  2025-09
              LYON PART DIEU            MARSEILLE ST CHARLES           3  2025-07  2025-09
              MACON LOCHE               PARIS LYON                     3  2025-07  2025-09
              PARIS EST                 STRASBOURG                     3  2025-07  2025-09
              MULHOUSE VILLE            PARIS LYON                     3  2025-07  2025-09
              MARSEILLE ST CHARLES      LYON PART DIEU                 3  2025-07  2025-09
              PARIS LYON                DIJON VILLE                    3  2025-07  2025-09
                                        CHAMBERY CHALLES LES EAUX      3  2025-07  2025-09
                                        BELLEGARDE (AIN)               3  2025-07  2025-09
                                        MACON LOCHE                    3  2025-07  2025-09
                                        MULHOUSE VILLE                 3  2025-07  2025-09
              STRASBOURG                PARIS EST                      3  2025-07  2025-09
National      BARCELONA                 PARIS LYON                    12  2025-10  2026-09
              BORDEAUX ST JEAN          PARIS VAUGIRARD               12  2018-01  2018-12
              PARIS VAUGIRARD           RENNES                        12  2018-01  2018-12
              RENNES                    PARIS VAUGIRARD               12  2018-01  2018-12
              PARIS VAUGIRARD           BORDEAUX ST JEAN              12  2018-01  2018-12
              PARIS LYON                BARCELONA                     12  2025-10  2026-09
              NANTES                    PARIS VAUGIRARD               12  2018-01  2018-12
              PARIS VAUGIRARD           NANTES                        12  2018-01  2018-12
              TOURCOING                 MARSEILLE ST CHARLES          13  2018-01  2022-08
International MADRID                    MARSEILLE ST CHARLES          45  2018-01  2022-12
              MARSEILLE ST CHARLES      MADRID                        45  2018-01  2022-12
              BARCELONA                 PARIS LYON                    93  2018-01  2025-09
              PARIS LYON                BARCELONA                     93  2018-01  2025-09
              ITALIE                    PARIS LYON                   101  2018-01  2026-09
              PARIS LYON                ITALIE                       101  2018-01  2026-09
National      BORDEAUX ST JEAN          TOURCOING                    103  2018-01  2026-09
              TOURCOING                 BORDEAUX ST JEAN             103  2018-01  2026-09
International PARIS LYON                ZURICH                       104  2018-01  2026-09
National      MARSEILLE ST CHARLES      TOURCOING                    104  2018-01  2026-09
International ZURICH                    PARIS LYON                   104  2018-01  2026-09

## Q2. Les noms de gares sont‑ils propres ?

In [12]:
gares = pd.Series(pd.concat([df["gare_de_depart"], df["gare_d_arrivee"]]).unique()).sort_values()
print("Nombre de gares :", len(gares))
print("Pas en majuscules :", gares[gares != gares.str.upper()].tolist())
print("Espaces en trop :", gares[gares != gares.str.split().str.join(" ")].tolist())
print("Avec accents :", gares[gares.map(lambda g: not g.isascii())].tolist())

Nombre de gares : 59
Pas en majuscules : []
Espaces en trop : []
Avec accents : []


In [13]:
# Variantes d'un même nom : on compare les noms réduits à leurs lettres et chiffres
cle = gares.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
gares.groupby(cle).agg(list)[lambda s: s.str.len() > 1]

Series([], dtype: object)

In [14]:
gares.tolist()

['AIX EN PROVENCE TGV',
 'ANGERS SAINT LAUD',
 'ANGOULEME',
 'ANNECY',
 'ARRAS',
 'AVIGNON TGV',
 'BARCELONA',
 'BELLEGARDE (AIN)',
 'BESANCON FRANCHE COMTE TGV',
 'BORDEAUX ST JEAN',
 'BREST',
 'CHAMBERY CHALLES LES EAUX',
 'DIJON VILLE',
 'DOUAI',
 'DUNKERQUE',
 'FRANCFORT',
 'GENEVE',
 'GRENOBLE',
 'ITALIE',
 'LA ROCHELLE VILLE',
 'LAUSANNE',
 'LAVAL',
 'LE CREUSOT MONTCEAU MONTCHANIN',
 'LE MANS',
 'LILLE',
 'LYON PART DIEU',
 'MACON LOCHE',
 'MADRID',
 'MARNE LA VALLEE',
 'MARSEILLE ST CHARLES',
 'METZ',
 'MONTPELLIER',
 'MULHOUSE VILLE',
 'NANCY',
 'NANTES',
 'NICE VILLE',
 'NIMES',
 'PARIS EST',
 'PARIS LYON',
 'PARIS MONTPARNASSE',
 'PARIS NORD',
 'PARIS VAUGIRARD',
 'PERPIGNAN',
 'POITIERS',
 'QUIMPER',
 'REIMS',
 'RENNES',
 'SAINT ETIENNE CHATEAUCREUX',
 'ST MALO',
 'ST PIERRE DES CORPS',
 'STRASBOURG',
 'STUTTGART',
 'TOULON',
 'TOULOUSE MATABIAU',
 'TOURCOING',
 'TOURS',
 'VALENCE ALIXAN TGV',
 'VANNES',
 'ZURICH']

## Q3. Les six pourcentages de causes somment‑ils à 100 ?

In [15]:
somme = df[CAUSES].sum(axis=1)
avec_retard = df[RET_ARR] > 0
print("Lignes avec au moins un train en retard :", avec_retard.sum())
print("  dont somme hors 100 +/- 1 :", ((somme - 100).abs() > 1)[avec_retard].sum())
print("  dont somme = 0 :", (somme == 0)[avec_retard].sum())
print("Lignes sans retard :", (~avec_retard).sum(), "-> somme des causes :", somme[~avec_retard].unique())
print("Pourcentages nuls (NaN) :", df[CAUSES].isna().sum().sum(),
      "/ négatifs :", (df[CAUSES] < 0).sum().sum(), "/ > 100 :", (df[CAUSES] > 100).sum().sum())

Lignes avec au moins un train en retard : 12715
  dont somme hors 100 +/- 1 : 89
  dont somme = 0 : 80
Lignes sans retard : 192 -> somme des causes : [0.]
Pourcentages nuls (NaN) : 0 / négatifs : 0 / > 100 : 0


In [16]:
# Où se trouvent les lignes en retard dont les causes valent toutes 0 ?
df[avec_retard & (somme == 0)]["date"].value_counts().sort_index()

date
2018-03     1
2018-05    17
2018-06    16
2018-07    16
2018-08    16
2019-09     1
2019-11     1
2020-04     4
2020-06     4
2023-09     1
2026-05     1
2026-07     1
2026-09     1
Name: count, dtype: int64

## Q4. Y a‑t‑il des valeurs impossibles ?

In [17]:
circules = df[PREV] - df[ANN]
controles = {
    "trains prévus = 0": df[PREV] == 0,
    "annulés > prévus": df[ANN] > df[PREV],
    "retard arrivée > circulés": df[RET_ARR] > circules,
    "retard départ > circulés": df[RET_DEP] > circules,
    "retard > 15 min > retard arrivée": df[R15] > df[RET_ARR],
    "retard > 30 min > retard > 15 min": df[R30] > df[R15],
    "retard > 60 min > retard > 30 min": df[R60] > df[R30],
    "retard > 30 min négatif": df[R30] < 0,
    "durée moyenne du trajet = 0": df["duree_moyenne_du_trajet"] == 0,
}
pd.DataFrame({
    "lignes": {k: v.sum() for k, v in controles.items()},
    "mois concernés": {k: ", ".join(sorted(df.loc[v, "date"].unique())[:6]) + (" ..." if df.loc[v, "date"].nunique() > 6 else "")
                       for k, v in controles.items()},
})

,lignes,mois concernés
trains prévus = 0,73,"2020-04, 2020-05, 2020-06, 2020-08, 2020-09, 2..."
annulés > prévus,63,"2020-04, 2020-05, 2020-06"
retard arrivée > circulés,63,"2020-04, 2020-05, 2020-06"
retard départ > circulés,64,"2020-04, 2020-05, 2020-06, 2022-06"
retard > 15 min > retard arrivée,31,"2018-01, 2018-02, 2018-03, 2019-05, 2019-12, 2..."
retard > 30 min > retard > 15 min,3,"2020-05, 2025-03"
retard > 60 min > retard > 30 min,354,"2025-01, 2025-02, 2025-03"
retard > 30 min négatif,43,"2025-01, 2025-02, 2025-03"
durée moyenne du trajet = 0,73,"2020-04, 2020-05, 2020-06, 2020-08, 2020-09, 2..."


Zoom sur le premier trimestre 2025 : la colonne « retard > 30 min » y paraît inexploitable.

In [18]:
t1_2025 = df["date"].between("2025-01", "2025-03")
print(f"T1 2025 : {(df[R60] > df[R30])[t1_2025].sum()} lignes sur {t1_2025.sum()} ont plus de retards > 60 min que > 30 min")
df.loc[t1_2025, KEY + [RET_ARR, R15, R30, R60]].head(8)

T1 2025 : 354 lignes sur 363 ont plus de retards > 60 min que > 30 min


,date,service,gare_de_depart,gare_d_arrivee,nombre_de_trains_en_retard_a_l_arrivee,nombre_trains_en_retard_15min,nombre_trains_en_retard_30min,nombre_trains_en_retard_60min
10324,2025-01,International,FRANCFORT,PARIS EST,27,27,0,18
10325,2025-01,International,GENEVE,PARIS LYON,19,19,0,13
10326,2025-01,International,LAUSANNE,PARIS LYON,15,15,-4,9
10327,2025-01,International,ZURICH,PARIS LYON,18,18,0,10
10328,2025-01,National,ANGOULEME,PARIS MONTPARNASSE,44,28,0,10
10329,2025-01,National,DIJON VILLE,PARIS LYON,60,47,-1,30
10330,2025-01,National,DOUAI,PARIS NORD,43,21,-1,9
10331,2025-01,National,DUNKERQUE,PARIS NORD,29,22,0,8


In [19]:
# Avril 2020 (confinement) : 0 train prévu mais des annulations, durée de trajet à 0
df.loc[df[PREV] == 0, KEY + [PREV, ANN, RET_ARR, "duree_moyenne_du_trajet"]].head(6)

,date,service,gare_de_depart,gare_d_arrivee,nombre_de_circulations_prevues,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,duree_moyenne_du_trajet
3406,2020-04,National,PARIS MONTPARNASSE,ST MALO,0,8,0,0
3407,2020-04,National,PARIS MONTPARNASSE,TOURS,0,9,0,0
3409,2020-04,National,NANTES,STRASBOURG,0,2,0,0
3411,2020-04,National,PARIS NORD,DOUAI,0,12,0,0
3412,2020-04,National,DUNKERQUE,PARIS NORD,0,15,0,0
3414,2020-04,National,MARSEILLE ST CHARLES,TOURCOING,0,1,0,0


In [20]:
# Retards moyens négatifs : trains en avance (plausible) ou valeurs aberrantes ?
retards_moyens = [c for c in df.columns if c.startswith("retard_moyen")]
(df[retards_moyens] < 0).sum().to_frame("lignes_negatives").join(df[retards_moyens].min().rename("minimum"))

,lignes_negatives,minimum
retard_moyen_des_trains_en_retard_au_depart,0,0.000000
retard_moyen_de_tous_les_trains_au_depart,170,-229.269444
retard_moyen_des_trains_en_retard_a_l_arrivee,2,-40.109259
retard_moyen_de_tous_les_trains_a_l_arrivee,122,-472.638889
retard_moyen_trains_en_retard_15_si_liaison_concurrencee_par_vol,8,-4.000000


In [21]:
# La colonne "retard moyen > 15 min (si liaison concurrencée par vol)" change de sens après 2018
col = "retard_moyen_trains_en_retard_15_si_liaison_concurrencee_par_vol"
df.groupby(df["date"].str[:4])[[col, "retard_moyen_de_tous_les_trains_a_l_arrivee",
                                "retard_moyen_des_trains_en_retard_a_l_arrivee"]].mean().round(1)

,retard_moyen_trains_en_retard_15_si_liaison_concurrencee_par_vol,retard_moyen_de_tous_les_trains_a_l_arrivee,retard_moyen_des_trains_en_retard_a_l_arrivee
date,,,
2018,7.1,7.1,32.9
2019,28.9,5.0,34.5
2020,35.6,4.6,29.3
2021,41.9,4.6,35.2
2022,42.3,6.4,35.3
2023,45.3,6.8,38.3
2024,43.3,6.3,36.1
2025,46.4,7.3,39.6
2026,47.4,9.2,41.0


## Q5. Quelles colonnes sont du texte libre ?

In [22]:
remplissage = df.groupby(df["date"].str[:4])[COMMENTAIRES].agg(lambda s: s.notna().mean()).round(2)
remplissage

,commentaire_annulations,commentaire_retards_au_depart,commentaire_retards_a_l_arrivee
date,,,
2018,0.0,0.0,0.29
2019,0.0,0.0,0.16
2020,0.0,0.0,0.00
2021,0.0,0.0,0.00
2022,0.0,0.0,0.00
2023,0.0,0.0,0.00
2024,0.0,0.0,0.00
2025,0.0,0.0,0.00
2026,0.0,0.0,0.00


In [23]:
multilignes = df["commentaire_retards_a_l_arrivee"].str.contains("\n", na=False).sum()
print("Commentaires sur plusieurs lignes :", multilignes)
print(df["commentaire_retards_a_l_arrivee"].dropna().iloc[0][:300])

Commentaires sur plusieurs lignes : 351
Ce mois-ci, l'OD a été touchée par les incidents suivants :
Le 1er : Tempête Carmen sur la façade Atlantique  (52 TGV ; 1079mn)
Le 3 : Tempête Eleonor sur l’ouest de la France (35 TGV ; 407mn)
Le 5 : Dérangement d’une aiguille en gare de Massy TGV (57 TGV ; 1254mn)
Le 7 : Dérangement du poste d’aigu


## Aperçu des indicateurs métier

Calcul rapide des indicateurs définis dans `docs/exploration.md`, pour vérifier qu'ils sont calculables.
Agrégation **pondérée** (somme des numérateurs / somme des dénominateurs), jamais une moyenne de taux.

In [24]:
annee = df.assign(
    annee=df["date"].str[:4],
    circules=(df[PREV] - df[ANN]).clip(lower=0),
    libelle=df["gare_de_depart"] + " > " + df["gare_d_arrivee"],
)
par_liaison = annee.groupby(["annee", "service", "libelle"]).agg(
    circules=("circules", "sum"), retards=(RET_ARR, "sum"), annules=(ANN, "sum"), prevus=(PREV, "sum"),
)
par_liaison["taux_ponctualite"] = 1 - par_liaison["retards"] / par_liaison["circules"]
par_liaison["taux_annulation"] = par_liaison["annules"] / par_liaison["prevus"]

derniere_annee_complete = sorted(d[:4] for d in df["date"].unique() if d.endswith("-12"))[-1]
(par_liaison.loc[derniere_annee_complete]
    .query("circules >= 200")
    .sort_values("taux_ponctualite")
    .head(10)
    .style.format({"taux_ponctualite": "{:.1%}", "taux_annulation": "{:.1%}"}))